# LLM Fine-Tuning Deep Dive: Data-Based vs Parameter-Based Techniques

This notebook is a technical deep dive into **LLM fine-tuning**, expanded from an earlier exploratory
notebook (`playground/af-advanced-ai/Unsupervised Finetuning Domain specific/non_Instruction_pretrain_llm_finetuning_on_domain_specific_data.ipynb`).

**Goal:** fine-tune a tiny, CPU-friendly base model (`distilgpt2`, ~82M parameters) on an original
multi-genre training corpus -- **seven complete novels spanning sci-fi, fantasy, mystery, historical
fiction, cyberpunk, horror, and literary fiction** (~422,300 words / ~2.57 MB total) -- so it learns
diverse vocabulary, narrative structures, and prose styles across genres. We use this corpus to
demonstrate **every major axis of fine-tuning**:

| Axis                | Question it answers                         | Techniques covered here                                                                           |
| ------------------- | ------------------------------------------- | ------------------------------------------------------------------------------------------------- |
| **Data-based**      | _What objective/data teaches the behavior?_ | Non-instructional (continued pretraining), Instructional (supervised), Preference alignment (DPO) |
| **Parameter-based** | _How many/which weights are updated?_       | Full fine-tuning, Partial (layer freezing), Parameter-efficient (LoRA)                            |

## Corpus

- **Location:** [`content/`](content/) -- **7 original novels** across diverse genres (sci-fi,
  fantasy, mystery, historical, cyberpunk, horror, literary), totaling **141 chapters** (~422,300
  words / ~2.57 MB). See [`content/README.md`](content/README.md) for the full breakdown and
  individual synopses.
- **Genres available:**
  - Sci-fi: _The Weight of Distant Light_ (generation ship, 32 chapters)
  - Fantasy: _The Tidebound Accord_ (epic quest, 25 chapters)
  - Mystery: _The Cartographer's Cipher_ (noir detective, 13 chapters)
  - Historical: _The Silk Merchant's Daughter_ (Tang Dynasty, 15 chapters)
  - Cyberpunk: _Neural Drift_ (memory broker conspiracy, 16 chapters)
  - Horror: _The Hollow Beneath_ (gothic/cosmic, 20 chapters)
  - Literary: _The Weight of Tides_ (marine biology first contact, 20 chapters)
- **Scale note:** Training cells sample a subset (`max_chapters` per genre) by default for fast CPU
  demos. Pass larger limits or `genres=None` to train on the full corpus.

## Setup

Run `setup.ps1` once to create a `.venv` and register the `llm-tuning` Jupyter kernel, then select
that kernel for this notebook.


In [9]:
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"  # ~82M params, real pretrained weights, fast enough to fine-tune on a CPU
# Use Path(__file__).parent if in .py, but in notebooks use absolute path or ensure content/ is in same dir
CONTENT_DIR = (
    Path(__file__).parent / "content"
    if "__file__" in dir()
    else Path("learning/genai/llm-tuning/content")
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Content directory: {CONTENT_DIR.absolute()}")

# Novel directory mappings (keys are shorthand aliases, values are actual directory names)
NOVELS = {
    "scifi": "the-weight-of-distant-light",
    "fantasy": "the-tidebound-accord",
    "mystery": "the-cartographers-cipher",
    "historical": "the-silk-merchants-daughter",
    "cyberpunk": "neural-drift",
    "horror": "the-hollow-beneath",
    "literary": "the-weight-of-tides",
}


def load_corpus_paragraphs(novels=None, max_chapters=4, min_len=200):
    """Load paragraphs from the multi-novel corpus for quick CPU demos.

    Args:
        novels: List of novel aliases to load (e.g., ["scifi", "fantasy"]), or None to
                load all. Available aliases: "scifi", "fantasy", "mystery", "historical",
                "cyberpunk", "horror", "literary" (mapped to directory names).
        max_chapters: Max chapters to load per novel (keeps CPU training fast).
        min_len: Skip paragraphs shorter than this many characters.

    Returns:
        List of paragraph strings from all requested novels.
    """
    if novels is None:
        novels = list(NOVELS.keys())  # load all by default

    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            print(f"Warning: unknown novel alias '{alias}', skipping")
            continue

        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            print(f"Warning: directory {novel_path} not found, skipping")
            continue

        chapter_files = sorted(novel_path.glob("chapter_*.txt"))[:max_chapters]
        for path in chapter_files:
            text = path.read_text(encoding="utf-8")
            for para in text.split("\n\n"):
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:
                    paragraphs.append(para)

    return paragraphs


def tokenize_causal(examples, tokenizer, max_length=128):
    """Standard next-token-prediction tokenization: labels = input_ids, with padding
    positions masked out (-100) so the loss ignores them."""
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length
    )
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens


sample_paragraphs = load_corpus_paragraphs(novels=["scifi", "fantasy"], max_chapters=2)
print(f"Loaded {len(sample_paragraphs)} sample paragraphs from 2 novels. First one:\n")
print(sample_paragraphs[0][:400], "...")

Using device: cpu
Content directory: c:\repos\ai-portfolio\learning\genai\llm-tuning\content
Loaded 127 sample paragraphs from 2 novels. First one:

Aria Voss had learned to listen to the ship the way other people listened to weather. Two hundred and fourteen years after the Meridian's Promise had folded its sails of solar canvas and slipped away from the blue marble of Earth, the great vessel still spoke in a thousand small voices, and Aria had spent her whole life learning to tell them apart. The groan of the spinward bearings when the ring  ...


## Why Fine-Tuning? The Three-Gap Problem

A pretrained LLM (like GPT-2, LLaMA, or Mistral) has learned language from billions of tokens of web
text, books, and code. This gives it strong **general fluency** and **broad world knowledge**. But for
any specific application, it has three concrete gaps:

### Gap 1: Domain Knowledge Gap

**Problem:** The model has never seen your specific vocabulary, characters, facts, or house style.

**Example with our corpus:**

- **You ask:** `"Who is Aria Voss?"`
- **Base model:** `"Aria Voss is a... [makes up something generic or says 'I don't know']"`
- **After fine-tuning:** `"Aria Voss is the Hold systems technician aboard the Meridian's Promise 
generation ship..."`

**Solution:** Continued pretraining on domain-specific text.

---

### Gap 2: Behavior Gap

**Problem:** A raw pretrained model just continues text. It doesn't know how to follow instructions,
answer questions directly, or stop when it should.

**Example:**

- **You ask:** `"List the five tides in the Tidebound Accord."`
- **After domain pretraining:** `"List the five tides in the Tidebound Accord. This question has 
puzzled scholars for millennia. Some say there are actually six tides, while others..."` (rambles
  forever)
- **After instruction tuning:** `"The five tides are: water, wind, stone, flame, and void."`

**Solution:** Instruction tuning (supervised fine-tuning) on (prompt, completion) pairs.

---

### Gap 3: Preference Gap

**Problem:** Even an instruction-following model may produce outputs that are technically correct but
not what humans actually prefer (too verbose, wrong tone, unhelpful focus).

**Example:**

- **You ask:** `"Explain quantum entanglement simply."`
- **After instruction tuning:** `"Quantum entanglement is a phenomenon in quantum mechanics wherein 
the quantum states of two or more particles become interdependent such that the state of one cannot 
be fully described without reference to the others, even when separated by large distances..."` (10
  more paragraphs of jargon)
- **After preference alignment:** `"Quantum entanglement means two particles become connected so 
measuring one instantly affects the other, even across vast distances."`

**Solution:** Preference alignment (RLHF or DPO) using human preference data.

---

### The Journey, Not a Taxonomy

These aren't three alternatives—they're **three sequential stages**. A production LLM pipeline
typically looks like:

```
Pretrained base ---> Continued pretraining ---> Instruction tuning ---> Preference alignment ---> Production model
    (Gap 0)               (Closes Gap 1)              (Closes Gap 2)            (Closes Gap 3)
```

At each stage, you also choose **how many parameters to update** (full fine-tuning vs. freezing vs.
LoRA), which we'll explore after demonstrating the data-based journey.

---

## Baseline: What Does the Un-Tuned Model Know?

Before fine-tuning, let's see what `distilgpt2` (pretrained on generic web text) produces when
prompted with a scenario from our sci-fi novel. Since it has never seen this story, expect a fluent
but generic, off-world continuation.


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"  # from the sci-fi corpus


def generate(model, prompt, max_new_tokens=60):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


print("=== Baseline (no fine-tuning) ===")
print(generate(base_model, PROMPT))

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

=== Baseline (no fine-tuning) ===
Aria Voss stared at the signal counting itself out in prime numbers and thought, ‪I can‪t wait for you to go home.‪

‪I‪ll leave you alone.‪
‪I don't even know what happened.‪
‪I don't even know what happened.‪
�


## Test Prompts for Validating Fine-Tuning

Use these prompts to test whether fine-tuning successfully absorbed domain-specific knowledge from
the seven-novel corpus. A well-tuned model should recognize characters, settings, and continue
narratives in the appropriate style. The baseline model (pretrained only) should produce generic,
off-topic continuations.

### Character & Setting Recognition Tests

**Sci-Fi (The Weight of Distant Light):**

- `"Aria Voss checked the Meridian's Promise status panel and"`
- `"The Keeper's consciousness flickered through node seventeen as"`
- `"In the Under-Hold, Nyla Kade whispered about the prime number signal from"`

**Fantasy (The Tidebound Accord):**

- `"Kerra Valmont felt all five tides simultaneously—water, wind, stone, flame, and void—as"`
- `"The ancient pillars rose from the Abyssal Rift while Davin Shale"`
- `"The Hollow King's followers, called the Hollowed,"`

**Mystery (The Cartographer's Cipher):**

- `"Elena Voss studied the 1879 survey map and realized the Ashmont Trust"`
- `"Detective Chen examined Adelaide Thorne's body and found the message: 'the foundation must hold'"`
- `"The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—"`

**Historical (The Silk Merchant's Daughter):**

- `"Wei Lian's jade phoenix pendant caught the morning light in Chang'an as"`
- `"Zhang Ming, the jinshi degree holder, wrote in his letter"`
- `"In the Eastern Market, the Wei family silk compound"`

**Cyberpunk (Neural Drift):**

- `"Kai Chen adjusted the neurorig and prepared to extract the memory backup from"`
- `"In the Lower Stacks of Neo-Shanghai, the stolen neural backups from Project Drift"`
- `"Victor Tang's consciousness transfer protocol failed when"`

**Horror (The Hollow Beneath):**

- `"Eleanor Vance sealed the cellar door at sunset, knowing that Blackwood Manor"`
- `"The hollow beneath the house breathed, and the entity in the limestone caves"`
- `"Margot found Thaddeus Blackwood's journal warning: never descend past the second chamber"`

**Literary (The Weight of Tides):**

- `"Claire Merritt opened her father's blue folder and read the July 12, 1975 entry about"`
- `"In Willowport, the underwater object near Whitehead Island caused"`
- `"The lobster traps came up bent, and the water temperature dropped fifteen degrees when"`

### Genre Style Continuation Tests

**Sci-Fi narrative momentum:**

- `"Two hundred and fourteen years after the Meridian's Promise left Earth,"`

**Fantasy elemental magic:**

- `"The tide-weavers gathered at Deepwater Crossing as the fifth tide, the void tide,"`

**Mystery noir atmosphere:**

- `"The rain-slicked streets of Ashmont Bay hid secrets from 1879, and Elena Voss"`

**Historical detail & restraint:**

- `"The silk road brought more than trade goods to Tang Dynasty Chang'an—it brought"`

**Cyberpunk tech-noir:**

- `"Memory extraction left traces, neural signatures that couldn't be scrubbed, and Kai Chen"`

**Gothic horror tension:**

- `"The house chose its inhabitants through grief, calling them when they were most vulnerable, and"`

**Literary introspection:**

- `"The ocean held its own memory, deeper and older than human documentation, and Claire"`

### Cross-Novel Vocabulary Tests

These should work across multiple genres if fine-tuning absorbed the corpus style:

- `"The weight of distant"` (tests sci-fi novel phrase bleed)
- `"The tidebound"` (tests fantasy terminology)
- `"permanent removal"` (tests mystery euphemism)
- `"steel in your spine, even if you must hide it beneath"` (tests historical voice)
- `"neural backup"` (tests cyberpunk jargon)
- `"the hollow"` (tests horror atmospheric language)
- `"The water's wrong"` (tests literary marine biology voice)


In [11]:
# Automated test runner: compare baseline vs fine-tuned on corpus-specific prompts
TEST_PROMPTS = {
    "scifi_character": "Aria Voss checked the Meridian's Promise status panel and",
    "fantasy_magic": "Kerra Valmont felt all five tides simultaneously—water, wind, stone, flame, and void—as",
    "mystery_conspiracy": "The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—",
    "historical_setting": "Wei Lian's jade phoenix pendant caught the morning light in Chang'an as",
    "cyberpunk_tech": "Kai Chen adjusted the neurorig and prepared to extract the memory backup from",
    "horror_atmosphere": "Eleanor Vance sealed the cellar door at sunset, knowing that Blackwood Manor",
    "literary_marine": "Claire Merritt opened her father's blue folder and read the July 12, 1975 entry about",
}


def test_corpus_knowledge(model, test_prompts=TEST_PROMPTS, max_new_tokens=50):
    """Run all test prompts and return results dict for comparison."""
    results = {}
    for key, prompt in test_prompts.items():
        results[key] = generate(model, prompt, max_new_tokens=max_new_tokens)
    return results


# Run baseline tests (will show generic, off-corpus continuations)
print(
    "=== BASELINE MODEL (no fine-tuning) - should produce generic continuations ===\n"
)
baseline_results = test_corpus_knowledge(base_model)
for key, output in baseline_results.items():
    print(f"[{key}]")
    print(output[:200] + "...\n")

=== BASELINE MODEL (no fine-tuning) - should produce generic continuations ===

[scifi_character]
Aria Voss checked the Meridian's Promise status panel and found no sign of an error.










































...

[fantasy_magic]
Kerra Valmont felt all five tides simultaneously—water, wind, stone, flame, and void—as it moved through the ocean, as it moved through the ocean.




































...

[mystery_conspiracy]
The six founding families—Ashmont, Thorne, Blackwell, Winters, Kahale, and Mordecai—have been in the midst of a war, which is over in the United States.
































...

[historical_setting]
Wei Lian's jade phoenix pendant caught the morning light in Chang'an as she was leaving home. The jade pendant had been caught, but the jade pendant had been caught by the jade pendant.


The jade pen...

[cyberpunk_tech]
Kai Chen adjusted the neurorig and prepared to extract the memory backup from her laptop computer. After the ini

Notice the output has no awareness of Aria Voss (from _The Weight of Distant Light_), the _Meridian's
Promise_, the Lantern, or any of the other characters/worlds across the seven novels -- it is fluent
English but a generic, unrelated continuation. This is exactly the gap fine-tuning closes.


---

## The Fine-Tuning Journey: A Problem-Solution Narrative

Rather than a flat taxonomy, fine-tuning is best understood as a **journey where each technique 
solves a problem left by the previous one**:

```mermaid
flowchart TD
    A[Pretrained Base Model] -->|Problem: Doesn't know your domain| B[Solution: Continued Pretraining]
    B -->|Problem: Continues text, won't follow instructions| C[Solution: Instruction Tuning SFT]
    C -->|Problem: Outputs aren't what humans prefer| D[Solution: Preference Alignment DPO/RLHF]
    
    style A fill:#e1f5ff
    style B fill:#b3e5fc
    style C fill:#81d4fa
    style D fill:#4fc3f7
```

At each stage, you can choose **how many parameters to update**:

| Approach | Trade-off | Use when |
|----------|-----------|----------|
| **Full fine-tuning** (100% params) | Max quality, max cost | Small models, abundant compute |
| **Partial freezing** (10-30% params) | Middle ground | Limited compute budget |
| **LoRA** (well under 1% params) | Min cost, swappable adapters | Most production scenarios |

**This notebook demonstrates:**
- All 3 data-based stages (continued pretraining, instruction tuning, preference alignment)
- All 3 parameter-based approaches (full, partial, LoRA)
- **Not covered** (mentioned for completeness): PPO-based RLHF, adapters, prefix tuning, QLoRA

---


## Concept 1 (Data-Based): Non-Instructional Fine-Tuning (Continued Pretraining)

**What it is:** keep training with the exact same objective used for the original pretraining --
next-token prediction -- but on your own raw, unlabeled domain text instead of general web text. No
prompts, no "instructions", no labeled pairs: just plain paragraphs. This is often called _continued
pretraining_ or _domain-adaptive pretraining (DAPT)_.

**When to use it:** you have a pile of domain text (support tickets, legal filings, a fictional
universe...) and you want the model to _absorb_ its vocabulary, facts, and style before you ever teach
it to follow instructions.

**Pros**

- Cheapest data to acquire -- no labeling/annotation needed, just clean text.
- Great at absorbing vocabulary, entities, and stylistic quirks (character names, invented
  terminology...).
- Simple training loop -- identical to pretraining (`labels = input_ids`).

**Cons**

- Does **not** teach the model to follow instructions or hold a conversation -- it only gets better
  at _continuing_ text like your domain text.
- Risk of shallow memorization instead of generalization if the corpus is small or repetitive.
- Risk of **catastrophic forgetting** of general-purpose ability if trained too long/aggressively.

Below we run this on a sample of chapters from the multi-genre corpus, updating **all** of
`distilgpt2`'s parameters (full fine-tuning -- more on that axis further down).


In [5]:
from datasets import Dataset
from transformers import Trainer, TrainingArguments

non_inst_paragraphs = load_corpus_paragraphs(
    novels=["scifi", "fantasy", "mystery"], max_chapters=3
)
print(
    f"Loaded {len(non_inst_paragraphs)} paragraphs from 3 novels for continued-pretraining demo"
)

non_inst_dataset = Dataset.from_dict({"text": non_inst_paragraphs})
non_inst_tokenized = non_inst_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)

full_ft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

training_args_full = TrainingArguments(
    output_dir="./checkpoints/non-instruction-full",
    per_device_train_batch_size=2,
    max_steps=25,  # keep the CPU demo fast; raise this (or drop max_steps) for real runs
    logging_steps=5,
    save_strategy="no",
    learning_rate=5e-5,
    report_to="none",
)

trainer_full = Trainer(
    model=full_ft_model, args=training_args_full, train_dataset=non_inst_tokenized
)
trainer_full.train()
full_ft_model.save_pretrained("./checkpoints/non-instruction-full")
print("Saved continued-pretraining (full fine-tune) checkpoint.")

Loaded 304 paragraphs from 3 novels for continued-pretraining demo


Map:   0%|          | 0/304 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

c:\repos\ai-portfolio\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,4.403024
10,4.308256
15,4.312697
20,4.297858
25,4.650691


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved continued-pretraining (full fine-tune) checkpoint.


## Concept 2 (Data-Based): Instructional (Supervised) Fine-Tuning

### The Problem with Continued Pretraining Alone

After continued pretraining, the model knows your domain vocabulary and can continue text in your
style. But try asking it a question:

**You:** `"What are the five tides in the Tidebound world?"`  
**Model (after continued pretraining):** `"What are the five tides in the Tidebound world? This 
question has puzzled scholars for centuries. Some believe there are actually six tides, while..."`
(continues rambling)

**The problem:** The model learned to _continue_ prose, not to _answer questions_ or _follow
instructions_. It will keep generating narrative-style text forever because that's what it was trained
on.

### The Solution: Instruction Tuning (Supervised Fine-Tuning / SFT)

**What it is:** Train on `(prompt, completion)` pairs where the **prompt** is an instruction/question
and the **completion** is the desired response. Crucially, we **mask the prompt tokens** in the loss
so the model is only penalized for the completion portion.

**Key insight:** This teaches the model _behavior_ -- "when you see input shaped like X, respond like
Y" -- rather than just "keep talking like this corpus."

Real-world instruction datasets include:

- [alpaca-cleaned](https://huggingface.co/datasets/yahma/alpaca-cleaned) - 52K instruction-following examples
- [OpenOrca](https://huggingface.co/datasets/Open-Orca/OpenOrca) - 4.2M GPT-4 completions
- [OpenAssistant/oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1) - 161K human-rated conversations

Here we auto-derive a tiny instruction dataset from our corpus:

- **Prompt:** `"Continue the fiction narrative in the same style: <paragraph N>"`
- **Completion:** `<paragraph N+1>`

**Pros:**

- Model learns to _follow a format/instruction_, not just continue prose
- Directly usable for chat/assistant interfaces
- Loss masking means the model isn't penalized for "predicting" the prompt it didn't generate

**Cons:**

- Needs actual (prompt, completion) pairs (expensive to create by hand)
- Can narrow diversity toward the exact template it was trained on
- Doesn't fix preference issues (model might follow instructions but in an unhelpful way)

This cell introduces **LoRA** (parameter-efficient tuning) to keep training fast on CPU.


In [12]:
from peft import LoraConfig, get_peft_model, TaskType

INSTRUCTION_PREFIX = "Continue the fiction narrative in the same style:\n\n"


def build_instruction_pairs(novels=None, max_chapters=3):
    if novels is None:
        novels = ["scifi", "fantasy"]  # default to 2 novels for demo speed

    pairs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias, "the-weight-of-distant-light")
        novel_path = CONTENT_DIR / novel_dir
        for path in sorted(novel_path.glob("chapter_*.txt"))[:max_chapters]:
            paras = [
                p.strip().replace("\n", " ")
                for p in path.read_text(encoding="utf-8").split("\n\n")
                if len(p.strip()) > 200
            ]
            for a, b in zip(paras, paras[1:]):
                pairs.append(
                    {"prompt": f"{INSTRUCTION_PREFIX}{a}\n\n", "completion": b}
                )
    return pairs


def tokenize_instruction(example, max_length=160, prompt_max_length=96):
    prompt_ids = tokenizer(
        example["prompt"], truncation=True, max_length=prompt_max_length
    )["input_ids"]
    full_text = example["prompt"] + example["completion"]
    tokens = tokenizer(
        full_text, truncation=True, padding="max_length", max_length=max_length
    )
    labels = tokens["input_ids"].copy()
    for i in range(min(len(prompt_ids), len(labels))):
        labels[i] = -100  # don't compute loss on the prompt portion
    for i, mask in enumerate(tokens["attention_mask"]):
        if mask == 0:
            labels[i] = -100  # don't compute loss on padding either
    tokens["labels"] = labels
    return tokens


instruction_pairs = build_instruction_pairs(
    novels=["scifi", "fantasy", "cyberpunk"], max_chapters=2
)
print(f"Built {len(instruction_pairs)} instruction pairs from 3 novels")

instruction_dataset = Dataset.from_list(instruction_pairs)
instruction_tokenized = instruction_dataset.map(
    tokenize_instruction, remove_columns=["prompt", "completion"]
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],  # GPT-2's combined attention projection
    lora_dropout=0.05,
    bias="none",
)

instruct_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = get_peft_model(instruct_base, lora_config)
instruct_lora_model.print_trainable_parameters()

training_args_instruct = TrainingArguments(
    output_dir="./checkpoints/instruction-lora",
    per_device_train_batch_size=2,
    max_steps=25,
    logging_steps=5,
    save_strategy="no",
    learning_rate=2e-4,
    report_to="none",
)

trainer_instruct = Trainer(
    model=instruct_lora_model,
    args=training_args_instruct,
    train_dataset=instruction_tokenized,
)
trainer_instruct.train()
instruct_lora_model.save_pretrained("./checkpoints/instruction-lora")
print("Saved instruction-tuned LoRA adapter.")

Built 179 instruction pairs from 3 novels


Map:   0%|          | 0/179 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

c:\repos\ai-portfolio\.venv\Lib\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


Step,Training Loss
5,4.217160
10,4.241254
15,4.511802
20,4.261203
25,4.481573


Saved instruction-tuned LoRA adapter.


## Concept 3 (Data-Based): Preference Alignment (RLHF / DPO)

### The Problem with Instruction Tuning Alone

After instruction tuning, the model follows instructions. But it might produce outputs that are
_technically correct_ but not what humans actually want:

**You:** `"Explain quantum entanglement."`  
**Instruction-tuned model:** `"Quantum entanglement is a phenomenon where particles become correlated 
such that the quantum state of one particle cannot be described independently... [continues for 10 
paragraphs with excessive jargon]"`

**Problems:**

- Too verbose (you wanted a 2-sentence explanation)
- Wrong tone (too academic for a casual question)
- Doesn't prioritize what you care about

**The core insight:** Instruction tuning teaches the model to _respond_, but not which responses
humans _prefer_.

### The Solution: Preference Alignment

**The idea:** Show the model pairs of responses to the same prompt -- one that humans prefer
(`chosen`) and one they don't (`rejected`) -- and train it to increase the probability of preferred
responses.

**Two approaches:**

1. **RLHF (Reinforcement Learning from Human Feedback):** Train a separate reward model to score
   responses, then use PPO (reinforcement learning) to optimize the LLM against that reward. (Complex,
   not demoed here.)
2. **DPO (Direct Preference Optimization):** Skip the reward model entirely and optimize directly
   against preference pairs with a closed-form loss. (Simpler, demoed below.)

Real-world preference datasets:

- [Anthropic/hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf) - 170K human preferences on
  helpfulness/harmlessness
- [ultrafeedback-binarized-preferences-cleaned](https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned)
  - 60K preference pairs

**Our synthetic preference data:**

- **Chosen:** The paragraph that actually follows in the novel (coherent, on-topic continuation)
- **Rejected:** An unrelated paragraph from a different chapter (off-topic, worse continuation)

This is a stand-in for real human preference labels, but demonstrates the mechanics.

### DPO: The Intuition Before the Math

**What we want:**

1. **Increase** the probability the model assigns to `chosen` responses
2. **Decrease** the probability for `rejected` responses
3. **Don't drift too far** from the original instruction-tuned model (to avoid mode collapse)

**How DPO achieves this:**

- Compare the model's log-probability for chosen vs. rejected
- Compare those to a frozen "reference" copy of the model (prevents drift)
- Use a sigmoid to convert the difference into a 0-1 probability
- Optimize with cross-entropy loss

**The formula** (sigma = sigmoid, beta = temperature controlling preference signal strength):

$$
\mathcal{L}_{DPO} = -\log \sigma\Big(\beta\big[(\log \pi_\theta(y_w \mid x) - \log \pi_{ref}(y_w
\mid x)) - (\log \pi_\theta(y_l \mid x) - \log \pi_{ref}(y_l \mid x))\big]\Big)
$$

**Breaking it down:**

- $\pi_\theta$ = current policy (model being trained)
- $\pi_{ref}$ = frozen reference policy (baseline)
- $y_w$ = chosen (winning) response, $y_l$ = rejected (losing) response
- **Inner term:** How much more does our model prefer `chosen` over `rejected` compared to the
  reference?
- **Sigmoid:** Squash that to a probability
- **Negative log:** Turn it into a loss we can minimize

**Pros:**

- No separate reward model needed (unlike PPO-based RLHF)
- Works well with parameter-efficient methods (cheap to iterate)
- Directly optimizes for human preferences

**Cons:**

- Needs paired preference data (expensive to collect at scale)
- Can over-optimize ("reward hacking") if beta is too high or data is noisy
- Requires keeping a frozen reference model in memory during training

The training loop below is a **simplified, from-scratch implementation** so you can see the mechanics.
Production code would use `trl.DPOTrainer`.


In [13]:
import copy
import torch.nn.functional as F


def build_preference_pairs(novels=None, max_chapters=4, max_pairs=15):
    if novels is None:
        novels = ["scifi", "fantasy", "mystery"]  # default mix

    chapter_files = []
    for alias in novels:
        novel_dir = NOVELS.get(alias, "the-weight-of-distant-light")
        novel_path = CONTENT_DIR / novel_dir
        chapter_files.extend(sorted(novel_path.glob("chapter_*.txt"))[:max_chapters])

    all_paragraphs = []
    for path in chapter_files:
        paras = [
            p.strip().replace("\n", " ")
            for p in path.read_text(encoding="utf-8").split("\n\n")
            if len(p.strip()) > 200
        ]
        all_paragraphs.append(paras)

    pairs = []
    for c_idx, paras in enumerate(all_paragraphs):
        other_chapter = all_paragraphs[(c_idx + 1) % len(all_paragraphs)]
        for i in range(len(paras) - 1):
            prompt = f"{INSTRUCTION_PREFIX}{paras[i]}\n\n"
            chosen = paras[i + 1]  # the real, on-topic continuation
            rejected = other_chapter[
                i % len(other_chapter)
            ]  # an unrelated paragraph -> a worse continuation
            pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    return pairs[:max_pairs]


def encode_response(prompt, response, max_length=160, prompt_max_length=96):
    """Tokenize prompt+response and return a response_mask marking only the response
    tokens (excluding the shared prompt and any padding) as targets for logprob math."""
    prompt_ids = tokenizer(prompt, truncation=True, max_length=prompt_max_length)[
        "input_ids"
    ]
    full = tokenizer(
        prompt + response, truncation=True, padding="max_length", max_length=max_length
    )
    response_mask = [0] * max_length
    start = min(len(prompt_ids), max_length)
    end = min(sum(full["attention_mask"]), max_length)
    for i in range(start, end):
        response_mask[i] = 1
    return {
        "input_ids": torch.tensor(full["input_ids"]).unsqueeze(0).to(device),
        "attention_mask": torch.tensor(full["attention_mask"]).unsqueeze(0).to(device),
        "response_mask": torch.tensor(response_mask).unsqueeze(0).to(device),
    }


def sequence_logprob(model, input_ids, attention_mask, response_mask):
    """Sum of log P(token_t | tokens<t) over the response-mask positions only."""
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]
    targets = input_ids[:, 1:]
    mask = response_mask[:, 1:]
    log_probs = F.log_softmax(logits, dim=-1)
    token_logprobs = torch.gather(log_probs, 2, targets.unsqueeze(-1)).squeeze(-1)
    return (token_logprobs * mask).sum(dim=-1)


BETA = 0.1
policy_model = instruct_lora_model  # continue tuning the instruction-tuned LoRA adapter
reference_model = copy.deepcopy(policy_model).eval()
for p in reference_model.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    [p for p in policy_model.parameters() if p.requires_grad], lr=1e-5
)

preference_pairs = build_preference_pairs(
    novels=["scifi", "fantasy", "horror"], max_chapters=3, max_pairs=15
)
print(f"Built {len(preference_pairs)} preference pairs from 3 novels for the DPO demo")

policy_model.train()
for step, pair in enumerate(preference_pairs):
    chosen = encode_response(pair["prompt"], pair["chosen"])
    rejected = encode_response(pair["prompt"], pair["rejected"])

    policy_chosen_lp = sequence_logprob(policy_model, **chosen)
    policy_rejected_lp = sequence_logprob(policy_model, **rejected)
    with torch.no_grad():
        ref_chosen_lp = sequence_logprob(reference_model, **chosen)
        ref_rejected_lp = sequence_logprob(reference_model, **rejected)

    logits = BETA * (
        (policy_chosen_lp - ref_chosen_lp) - (policy_rejected_lp - ref_rejected_lp)
    )
    loss = -F.logsigmoid(logits).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 5 == 0:
        print(f"step {step:02d} | dpo_loss={loss.item():.4f}")

policy_model.save_pretrained("./checkpoints/preference-dpo")
print("Saved DPO-aligned adapter.")

Built 15 preference pairs from 3 novels for the DPO demo
step 00 | dpo_loss=0.4772
step 05 | dpo_loss=0.3972
step 10 | dpo_loss=0.7140
Saved DPO-aligned adapter.


## Parameter-Based Axis: How Many Weights Do We Actually Update?

### The Cost Problem

All three data-based techniques (continued pretraining, instruction tuning, preference alignment) work
by gradient descent on model weights. But **updating all weights is expensive:**

- **Memory:** 82M parameters × (4 bytes per param + 8 bytes optimizer state) = ~1 GB just for
  distilgpt2. For 70B models, this becomes **840 GB**.
- **Compute:** More trainable params = longer training time
- **Risk:** Full updates can "overwrite" the model's general knowledge (catastrophic forgetting)

### The Trade-Off Spectrum

Independent of _what data_ you train on, you can choose _how much of the model_ to update:

| Technique            | Trainable % | Memory  | Quality | Forgetting Risk | When to Use                         |
| -------------------- | ----------- | ------- | ------- | --------------- | ----------------------------------- |
| **Full fine-tuning** | 100%        | Highest | Highest | Highest         | Small models, abundant compute      |
| **Partial freezing** | 10-30%      | Medium  | Medium  | Medium          | Limited budget, want more than PEFT |
| **LoRA**             | <1%         | Lowest  | High    | Lowest          | Most production scenarios today     |

The following cells apply all three strategies to the _same_ continued-pretraining objective, so
parameter counts are directly comparable.

---

### Concept 4 (Parameter-Based): Full Fine-Tuning

**What it is:** Every single weight in the model is unfrozen and updated by the optimizer.

**Pros:** The model has maximum "room" to adapt to your domain.

**Cons:**

- Costs the most memory/compute
- Highest risk of catastrophic forgetting if domain corpus is small
- For large models (>7B params), often infeasible without multi-GPU setups

We already ran this in Concept 1 (`./checkpoints/non-instruction-full`) -- that cell **is** full
fine-tuning. The cell below quantifies what "100% trainable" looks like for `distilgpt2`.


In [14]:
param_check_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
total = sum(p.numel() for p in param_check_model.parameters())
trainable = sum(p.numel() for p in param_check_model.parameters() if p.requires_grad)
print(
    f"Full fine-tuning: {trainable:,}/{total:,} parameters trainable ({trainable / total * 100:.1f}%)"
)
del param_check_model

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Full fine-tuning: 81,912,576/81,912,576 parameters trainable (100.0%)


### Concept 5 (Parameter-Based): Partial Fine-Tuning (Layer Freezing)

**The observation:** In transformer models, **early layers** learn general language features
(tokenization, basic syntax, common words) while **later layers** learn task-specific patterns. This
is similar to how early layers in CNNs detect edges, while later layers detect objects.

**The strategy:** Freeze everything, then selectively unfreeze:

- The last N transformer blocks (task-specific adaptation)
- The output head (final projection to vocabulary)

**Pros:**

- Much cheaper than full fine-tuning (only 10-30% of parameters)
- Less prone to catastrophic forgetting (general features preserved)
- No new architecture needed

**Cons:**

- Still edits raw model weights (can't easily "swap" like an adapter)
- Choosing _how many_ layers to unfreeze is a manual hyperparameter
- Middle ground: not as cheap as LoRA, not as powerful as full fine-tuning

**Example:** For `distilgpt2` (6 transformer blocks), we unfreeze only the last 2 blocks + output
head.


In [15]:
freeze_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

for param in freeze_model.parameters():
    param.requires_grad = False

n_layers = freeze_model.config.n_layer  # distilgpt2 has 6 transformer blocks
unfreeze_from = n_layers - 2  # unfreeze only the last 2 blocks

for name, param in freeze_model.named_parameters():
    if any(f"h.{i}." in name for i in range(unfreeze_from, n_layers)):
        param.requires_grad = True
    if "ln_f" in name or "lm_head" in name:
        param.requires_grad = True

trainable = sum(p.numel() for p in freeze_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in freeze_model.parameters())
print(
    f"Partial fine-tuning: {trainable:,}/{total:,} parameters trainable ({trainable / total * 100:.2f}%)"
)

freeze_dataset = Dataset.from_dict(
    {"text": load_corpus_paragraphs(novels=["fantasy", "cyberpunk"], max_chapters=3)}
)
freeze_tokenized = freeze_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)

training_args_freeze = TrainingArguments(
    output_dir="./checkpoints/partial-freeze",
    per_device_train_batch_size=2,
    max_steps=25,
    logging_steps=5,
    save_strategy="no",
    learning_rate=1e-4,
    report_to="none",
)

trainer_freeze = Trainer(
    model=freeze_model, args=training_args_freeze, train_dataset=freeze_tokenized
)
trainer_freeze.train()
freeze_model.save_pretrained("./checkpoints/partial-freeze")
print("Saved partial (layer-freezing) fine-tune checkpoint.")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Partial fine-tuning: 14,177,280/81,912,576 parameters trainable (17.31%)


Map:   0%|          | 0/215 [00:00<?, ? examples/s]

Step,Training Loss
5,4.399514
10,4.638628
15,4.577361
20,4.314101
25,4.409713


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved partial (layer-freezing) fine-tune checkpoint.


### Concept 6 (Parameter-Based): Parameter-Efficient Fine-Tuning (LoRA)

**The key insight:** Instead of updating existing weights, **freeze the entire base model** and inject
small trainable matrices alongside key weight matrices.

**How LoRA works:**

For a weight matrix $W$ (e.g., attention projection), instead of updating $W \rightarrow W + \Delta W$,
we:

1. **Freeze** $W$ (no updates ever)
2. **Add** a low-rank decomposition: $\Delta W = BA$ where:
   - $B$ is $d \times r$ (rank-reducing projection)
   - $A$ is $r \times d$ (rank-expanding projection)
   - $r \ll d$ (rank is much smaller than original dimension)

**Example:** For GPT-2's attention (d=768), with r=8:

- Original: 768 × 768 = **589,824 parameters**
- LoRA: (768×8) + (8×768) = **12,288 parameters** (2% of original)

**What to tune:**

- `r` (rank): Higher = more capacity but more parameters. Typical: 4-64.
- `target_modules`: Which weight matrices to adapt. For transformers: attention projections (`q`,
  `k`, `v`, `o`) and sometimes feed-forward layers.
- `lora_alpha`: Scaling factor (typical: 2×r)

**Pros:**

- **Tiny memory footprint:** Only adapter's optimizer state needed
- **Swappable:** Train multiple adapters on the same frozen base model, swap at inference
- **Mergeable:** Can merge $BA$ into $W$ for zero-latency deployment
- **Lowest forgetting risk:** Base weights never change

**Cons:**

- Slightly lower quality ceiling than full fine-tuning for extreme distribution shifts
- Adds hyperparameters to tune (`r`, `alpha`, `target_modules`)
- Inference needs adapter loaded/merged

**Related techniques not demoed here:**

- **Adapters:** Small bottleneck layers inserted between transformer blocks
- **Prefix tuning:** Learn virtual tokens prepended to input
- **QLoRA:** LoRA on top of 4-bit quantized base model (GPU-specific)

This cell applies LoRA to _continued pretraining_ (not instruction tuning) to show the parameter axis
and data axis are independent choices.


In [16]:
lora_config_pt = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
)

lora_pt_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
lora_pt_model = get_peft_model(lora_pt_base, lora_config_pt)
lora_pt_model.print_trainable_parameters()

lora_pt_dataset = Dataset.from_dict(
    {
        "text": load_corpus_paragraphs(
            novels=["mystery", "horror", "literary"], max_chapters=3
        )
    }
)
lora_pt_tokenized = lora_pt_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)

training_args_lora_pt = TrainingArguments(
    output_dir="./checkpoints/peft-lora",
    per_device_train_batch_size=2,
    max_steps=25,
    logging_steps=5,
    save_strategy="no",
    learning_rate=2e-4,
    report_to="none",
)

trainer_lora_pt = Trainer(
    model=lora_pt_model, args=training_args_lora_pt, train_dataset=lora_pt_tokenized
)
trainer_lora_pt.train()
lora_pt_model.save_pretrained("./checkpoints/peft-lora")
print("Saved parameter-efficient (LoRA) continued-pretraining adapter.")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


Map:   0%|          | 0/338 [00:00<?, ? examples/s]

Step,Training Loss
5,4.418751
10,4.648992
15,4.591954
20,4.467506
25,4.701658


Saved parameter-efficient (LoRA) continued-pretraining adapter.


## Comparing All Six Techniques

| Technique                                 | Axis      | Trainable Params (this notebook)   | Data Needed                        | Best For                                    |
| ----------------------------------------- | --------- | ---------------------------------- | ---------------------------------- | ------------------------------------------- |
| Non-instructional (continued pretraining) | Data      | 100% (full FT, as run above)       | Raw domain text                    | Absorbing vocabulary/style/facts            |
| Instructional (SFT)                       | Data      | well under 1% (LoRA, as run above) | (prompt, completion) pairs         | Teaching task-following behavior            |
| Preference alignment (DPO)                | Data      | well under 1% (LoRA, as run above) | (prompt, chosen, rejected) triples | Aligning to human preference                |
| Full fine-tuning                          | Parameter | 100%                               | Any of the above                   | Max quality, abundant compute               |
| Partial (layer freezing)                  | Parameter | roughly 10-30% (last N layers)     | Any of the above                   | Middle ground on compute/quality            |
| Parameter-efficient (LoRA)                | Parameter | well under 1%                      | Any of the above                   | Cheapest, swappable, lowest forgetting risk |

In production, a realistic pipeline stacks the data-based stages in order (continued pretraining then
instruction tuning then preference alignment) while picking whichever parameter-based technique fits
the compute budget at each stage -- most commonly LoRA throughout, given how large modern base models
are.

## Side-by-Side: Every Checkpoint on the Same Prompt

Finally, let's compare the baseline against every fine-tuned variant trained above, on the same prompt
from the corpus.


In [18]:
# Select a smaller subset of prompts for comparison across all models
COMPARISON_PROMPTS = {
    "scifi": "Aria Voss checked the Meridian's Promise status panel and",
    "fantasy": "Kerra Valmont felt all five tides simultaneously as",
    "mystery": "Elena Voss studied the 1879 survey map and realized",
    "cyberpunk": "In the Lower Stacks of Neo-Shanghai, Kai Chen",
}

# Load the continued pretraining checkpoint for comparison
non_instruct_ckpt = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/non-instruction-full"
).to(device)

models_to_test = {
    "Baseline (no fine-tuning)": base_model,
    "Continued pretraining (full FT)": non_instruct_ckpt,
    "Instruction-tuned (LoRA)": instruct_lora_model,
    "Preference-aligned (DPO)": policy_model,
    "Partial fine-tuning": freeze_model,
    "PEFT LoRA continued pretraining": lora_pt_model,
}

print("=" * 80)
print("CORPUS KNOWLEDGE COMPARISON ACROSS ALL FINE-TUNING TECHNIQUES")
print("=" * 80)

for prompt_name, prompt in COMPARISON_PROMPTS.items():
    print(f"\n{'─' * 80}")
    print(f'PROMPT ({prompt_name}): "{prompt}"')
    print(f"{'─' * 80}\n")

    for model_name, model in models_to_test.items():
        # For instruction-tuned models, prepend the instruction prefix
        if "Instruction" in model_name or "Preference" in model_name:
            test_prompt = INSTRUCTION_PREFIX + prompt + "\n\n"
        else:
            test_prompt = prompt

        output = generate(model, test_prompt, max_new_tokens=60)
        # Extract just the generated portion (remove prompt)
        generated = (
            output[len(test_prompt) :] if output.startswith(test_prompt) else output
        )

        print(f"[{model_name}]")
        print(generated[:150] + "..." if len(generated) > 150 else generated)
        print()

print("\n" + "=" * 80)
print("ANALYSIS:")
print("- Baseline should produce generic, off-corpus continuations")
print("- Fine-tuned models should recognize characters/settings and continue in-world")
print("- Compare vocabulary, narrative coherence, and genre-appropriate style")
print("=" * 80)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

CORPUS KNOWLEDGE COMPARISON ACROSS ALL FINE-TUNING TECHNIQUES

────────────────────────────────────────────────────────────────────────────────
PROMPT (scifi): "Aria Voss checked the Meridian's Promise status panel and"
────────────────────────────────────────────────────────────────────────────────

[Baseline (no fine-tuning)]
 found it was in fact a "very important" part of the agreement.



"As a result, the contract between the company and the government is still not sign...

[Continued pretraining (full FT)]
 found no indication that the planet had been in orbit for more than a month, until she called for the cancellation of the contract and the discovery ...

[Instruction-tuned (LoRA)]
The other two panels
The other panel
The other panel
The other panel
The other panel
The other panel
The other panel
The other panel
The other panel
T...

[Preference-aligned (DPO)]
Caleen, the storyteller, met with the writer.
The writer met with the writer.
Caleen, the storyteller, met with the w

## Automated Corpus Knowledge Tests: All Models

Now let's run the corpus-specific test prompts on every fine-tuned checkpoint and compare how well
each technique absorbed the domain knowledge. We'll use one representative prompt from each novel and
compare baseline vs. all fine-tuned variants.


In [19]:
print("=== Baseline (no fine-tuning) ===")
print(generate(base_model, PROMPT), "\n")

print("=== Non-instructional continued pretraining (full fine-tune) ===")
# non_instruct_ckpt already loaded in the comparison cell above
print(generate(non_instruct_ckpt, PROMPT), "\n")

print("=== Instruction-tuned (LoRA) ===")
print(generate(instruct_lora_model, INSTRUCTION_PREFIX + PROMPT + "\n\n"), "\n")

print("=== Preference-aligned (DPO on top of the instruction-tuned LoRA adapter) ===")
print(generate(policy_model, INSTRUCTION_PREFIX + PROMPT + "\n\n"), "\n")

print("=== Partial fine-tuning (layer freezing) ===")
print(generate(freeze_model, PROMPT), "\n")

print("=== Parameter-efficient (LoRA continued pretraining) ===")
print(generate(lora_pt_model, PROMPT), "\n")

=== Baseline (no fine-tuning) ===
Aria Voss stared at the signal counting itself out in prime numbers and the numbers just kept coming up. The problem was, this wasn't the case, this wasn't the case. This was the case.

The problem was, this wasn't the case. There was no way for a group to know exactly how many times the number of times the number of 

=== Non-instructional continued pretraining (full fine-tune) ===
Aria Voss stared at the signal counting itself out in prime numbers and it only turned out to be a random number. He couldn't believe it, he couldn't even see it. "No, the problem is that it didn't work. The problem is that it wasn't really the only one, but it didn't exist."



"Why didn 

=== Instruction-tuned (LoRA) ===
Continue the fiction narrative in the same style:

Aria Voss stared at the signal counting itself out in prime numbers and

Aria Voss was surprised by the fact that it wasn't a full set of numbers. She could not believe how hard it was to make it count.
B

## What This Notebook Covered (and What It Didn't)

### Implemented and Demonstrated

**Data-based progression (the journey):**

1. **Continued pretraining** - Absorb domain vocabulary, facts, and style
2. **Instruction tuning (SFT)** - Teach the model to follow instructions, not just continue text
3. **Preference alignment (DPO)** - Align outputs with human preferences beyond "technically correct"

**Parameter-based approaches (the cost/quality trade-off):**

1. **Full fine-tuning** (100% params) - Maximum quality, maximum cost
2. **Partial freezing** (10-30% params) - Middle ground
3. **LoRA** (<1% params) - Minimum cost, swappable adapters

**All combinations tested** on the same 7-novel corpus with side-by-side comparisons.

### Mentioned but Not Implemented

**Alternative preference alignment:**

- **PPO-based RLHF** with separate reward model (more complex than DPO)

**Alternative parameter-efficient methods:**

- **Adapter layers** (bottleneck modules between transformer blocks)
- **Prefix/prompt tuning** (learnable virtual tokens prepended to input)
- **QLoRA** (LoRA on 4-bit quantized models, GPU-specific)
- **BitFit** (bias-only tuning)
- **IA3** (learned rescaling vectors)

**Why these weren't included:**

- DPO is simpler and more practical than PPO-based RLHF
- LoRA has become the dominant PEFT method in production (2024-2026)
- Other PEFT methods offer different trade-offs but similar principles

---

## Further Reading & Scaling Up

**To scale this notebook:**

- **Larger corpus:** Set `max_chapters=None` to use all 141 chapters (~422K words)
- **More novels:** Add `.txt` files to `content/` and update the `NOVELS` dict
- **Bigger models:** Replace `distilgpt2` with `gpt2-medium`, `gpt2-large`, etc. (will need GPU)
- **Real datasets:**
  - Non-instructional: [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories),
    [openwebtext](https://huggingface.co/datasets/Skylion007/openwebtext)
  - Instructional: [alpaca-cleaned](https://huggingface.co/datasets/yahma/alpaca-cleaned),
    [OpenOrca](https://huggingface.co/datasets/Open-Orca/OpenOrca)
  - Preferences: [Anthropic/hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf),
    [ultrafeedback-binarized](https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned)

**Key papers:**

- LoRA: [Hu et al. 2021](https://arxiv.org/abs/2106.09685)
- DPO: [Rafailov et al. 2023](https://arxiv.org/abs/2305.18290)
- Instruction tuning: [Wei et al. 2021 (FLAN)](https://arxiv.org/abs/2109.01652)
- RLHF: [Ouyang et al. 2022 (InstructGPT)](https://arxiv.org/abs/2203.02155)
